# LV2. 의약품 그래프를 평가하고 청크 크기를 바꿔 비교합니다

**교안 02의 방법으로 LV1의 같은 문서를 평가합니다.**  
LV1의 `Drug -> HAS_SIDE_EFFECT -> Symptom` 규칙을 유지하며, 설정 변경이 이상반응 관계의 누락과 오류에 어떤 영향을 주는지 확인하세요.  

- **입력:** LV1의 `output/baseline_assignment.json`, 정답 목록 `data/assignment_gold.json`.
- **결과:** 스키마 준수율, 근거 원문 일치율, 정밀도, 재현율, F1과 FP/FN 목록. 새 결과는 `output/large_assignment.json`에 저장합니다.
- **변경:** 청크 크기만 300 -> 1000. 원문, 모델, 스키마, 지시와 겹침 80은 유지합니다.

| 과제 순서 | 적용할 내용 | 교안 02 |
|---|---|---|
| 1 | 평가할 저장 단계와 문서 확인 | 1절 |
| 2 | 전체 행에서 두 검사 | 2절 |
| 3 | 고정 골드와 고유 관계 비교 | 3절 |
| 4 | 청크 크기 변경과 재평가 | 4절 |

**점수가 올랐는지가 채점 기준은 아닙니다.** 같은 범위와 분모를 유지하고 실제 FP와 FN을 설명하는 것이 목표입니다.  
1~3절은 저장 파일만 사용합니다. DB와 모델은 4절에서 연결합니다.  
교안은 겹침 비율을 유지했지만, 이 과제는 겹침 80자를 고정해 청크 크기만 바꿉니다.  

#### 사용할 라이브러리 불러오기

정답과 학생 작성 영역에 필요한 라이브러리를 먼저 불러옵니다.  

In [ ]:
# [제공코드]
import json
from pathlib import Path
from pprint import pprint
from collections import Counter
from math import isclose
import os
from urllib.parse import urlsplit
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from copy import deepcopy
from functools import partial
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from neo4j_graphrag.experimental.components.text_splitters.langchain import (
    LangChainTextSplitterAdapter,
)
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from uuid import uuid4

#### 자료 경로 준비

먼저 LV1을 끝내고 기준 파일을 만드세요. 교안의 JSON 읽기와 저장 함수를 사용합니다.  

In [ ]:
# [제공코드]

# 학생용은 현재 폴더, 정답은 한 단계 위 폴더의 자료를 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(path):
    """JSON 파일 하나를 파이썬 사전 또는 목록으로 읽습니다."""
    # path는 파일 위치이며, UTF-8로 읽어 한글을 유지합니다.
    return json.loads(path.read_text(encoding="utf-8"))


def save_json(path, value):
    """실행 결과를 한글을 유지한 JSON 파일로 저장합니다."""
    # value는 저장할 사전이나 목록입니다. ensure_ascii=False는 한글을 문자 그대로 남깁니다.
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

## 1. 평가할 실행과 문서를 확인합니다

<img src="./images/data_files_and_evaluation_flow.png" width="1000" alt="원문 document JSON에서 text만 모델에 전달합니다. Neo4j의 추출 결과는 baseline 저장본으로 남기고, 원문을 검토해 만든 gold와 비교합니다. 설정 변경 결과 large도 같은 gold로 평가합니다.">

이 과제에서는 그림의 `*` 자리에 `assignment`를 넣으면 실제 파일 이름이 됩니다. LV1의 결과와 골드를 읽고, 재실행 결과는 다른 파일에 저장합니다.  

`task_snapshot`은 LV1에서 저장한 설정과 결과 전체입니다.  
`document`는 원문과 출처, `rows`는 스키마 검사 후, 노드 통합 전에 저장한 관계 목록입니다. 가지치기로 제외된 관계는 이 목록에 없습니다.  

#### 기준 결과 읽기

문서와 관계를 꺼내 평가 단계와 저장 행 수를 출력하세요.  

In [ ]:
# (1) baseline_assignment.json을 task_snapshot에 읽으세요.
# (2) document를 task_doc, rows를 task_rows에 담으세요.
# (3) stage와 문서 제목, 저장 관계 행 수를 출력하세요.
# 여기에 코드를 작성하세요.

#### 확인하기

LV1의 문서와 통합 전 저장 결과를 평가하는지 확인합니다.  

In [ ]:
# [제공코드]
assert task_doc["doc_id"] == "drug_198500050", (
    "의약품 데이터로 LV1을 다시 실행한 뒤 진행하세요"
)
assert task_rows == task_snapshot["rows"]
assert task_snapshot["perform_entity_resolution"] is False
assert task_snapshot["chunk_size"] == 300
assert task_snapshot["chunk_overlap"] == 80
assert task_doc == read_json(data_dir / "assignment_document.json"), (
    "과제 원문과 출처를 유지하세요"
)
print("LV1의 기준 실행을 확인했습니다.")

## 2. 같은 전체 행에서 스키마와 근거를 각각 검사합니다

- **분모:** 두 검사 모두 전체 저장 행 수인 `len(task_rows)`입니다. 검사 때문에 `task_rows`를 줄이지 않습니다.
- **오류 목록:** 근거 검사 실패 행을 `task_evidence_errors`에 담습니다. 원래 목록에서 실패한 각 행을 한 번씩 담고, 순서는 자유입니다.
- **중복:** 원래 저장본에 같은 트리플이 여러 행이면 여기서는 각각 셉니다. 3절의 골드 비교에서는 하나로 셉니다.

**저장 관계가 있으면 스키마 준수율은 100%가 정상입니다.** LV1의 빌더가 가지치기한 결과를 확인하는 검사입니다.  
근거 일치율과 골드 점수는 별도이며, 원문을 그대로 인용했어도 잘못된 관계일 수 있습니다.  

#### 검사 함수 준비

`schema_ok`는 타입 조합, `evidence_ok`는 출처와 인용 문자열을 검사합니다. `triple_key`는 주어, 관계, 목적어를 꺼냅니다.  

In [ ]:
# [제공코드]
def schema_ok(row):
    """관계 이름과 양 끝의 타입이 허용한 조합인지 검사합니다."""
    return (row["subject_type"], row["relation"], row["object_type"]) == (
        "Drug",
        "HAS_SIDE_EFFECT",
        "Symptom",
    )


def evidence_ok(row, document):
    """근거가 비어 있지 않고 같은 문서의 전체 원문에 그대로 있으면 True입니다.

    document는 원문 문서 사전입니다. 개별 청크는 검사하지 않으므로,
    다른 청크의 문장을 인용했는지나 관계의 의미가 맞는지는 판정하지 않습니다.
    """
    # (1) row는 저장 관계 한 행, document는 대조할 출처 문서입니다.
    if row["source_doc_id"] != document["doc_id"]:
        return False

    # (2) 문자열이 아니거나 공백뿐이면 인용문으로 인정하지 않습니다.
    evidence = row["evidence"]
    if not isinstance(evidence, str) or not evidence.strip():
        return False

    # (3) 전체 문서에서 인용 문자열을 찾습니다. 출처 청크 내부의 일치는 별도 검사입니다.
    return evidence in document["text"]


def triple_key(row):
    """정밀도와 재현율에서 한 관계로 세는 (주어, 관계, 목적어)를 반환합니다."""
    return row["subject"], row["relation"], row["object"]

#### 두 비율과 근거 오류 출력

분자와 분모를 함께 출력하세요. 원문 일치 검사에 실패한 행은 근거와 연결된 청크를 출력해 불일치 이유를 확인합니다.  

In [ ]:
# (1) task_rows 전체에서 schema_ok와 evidence_ok의 통과 수를 각각 구하세요.
# task_schema_count, task_evidence_count에 담고 전체 수는 task_total로 둡니다.
# (2) 각 분자와 분모, 백분율을 출력하세요. 행이 없으면 미산출로 표시하세요.
# (3) 근거 검사 실패 행을 task_evidence_errors에 담고 트리플, 근거, 출처 청크를 출력하세요.
# 여기에 코드를 작성하세요.

#### 확인하기

오류를 골랐어도 평가할 전체 행과 분모가 유지되는지 확인합니다.  

In [ ]:
# [제공코드]
# 파일에서 다시 읽어야 원래 목록을 직접 바꾼 경우도 발견할 수 있습니다.
original_rows = read_json(output_dir / "baseline_assignment.json")["rows"]
assert task_rows == original_rows, "검사 때문에 원래 행을 삭제하지 마세요"
assert task_total == len(original_rows)
assert task_schema_count == sum(schema_ok(row) for row in task_rows)
assert task_evidence_count == sum(evidence_ok(row, task_doc) for row in task_rows)
assert task_evidence_count + len(task_evidence_errors) == task_total
# 순서는 채점하지 않지만, 오류 행을 빠뜨리거나 같은 행을 더 넣으면 안 됩니다.
expected_errors = Counter()
for row in task_snapshot["rows"]:
    if not evidence_ok(row, task_doc):
        expected_errors[json.dumps(row, sort_keys=True)] += 1
assert (
    Counter(json.dumps(row, sort_keys=True) for row in task_evidence_errors)
    == expected_errors
)
print("전체 행에서 두 검사를 독립적으로 계산했습니다.")

## 3. 같은 문서의 골드로 정밀도, 재현율과 F1을 구합니다

`assignment_gold.json`은 원문 5개 절을 검토해 작성한 정답 5관계입니다.  
[이상반응] 절의 개별 증상이 정답입니다. 효능, 사용법, 주의사항, 상호작용은 이 관계의 정답 0건입니다. 골드는 추출 지시에 넣지 않습니다.  
같은 `(주어, 관계, 목적어)`가 여러 행에 있으면 하나로 세고, 검사 실패 행도 평가에 포함하세요.  

#### 정답 목록 준비

교안의 평가 함수 안에서 배운 집합 연산과 공식을 직접 적용합니다.  

In [ ]:
# [제공코드]

# 원문에서 작성한 고정 정답입니다. 청크 크기를 바꾼 뒤에도 같은 목록을 씁니다.
task_gold = read_json(data_dir / "assignment_gold.json")
print("골드 관계 수:", len(task_gold))

#### 고유 관계로 점수와 오류 구하기


`triple_key`로 만든 추출 집합은 `task_predicted`, 골드 집합은 `task_expected`에 담으세요.  
결과는 아래 키를 가진 사전 `task_metrics`로 만듭니다.  

| 키 | 담을 값 |
|---|---|
| `tp`, `fp`, `fn` | 각각 맞힌 관계, 잘못 뽑은 관계, 놓친 관계의 **집합** |
| `precision` | 맞힌 수 ÷ 고유 추출 수 |
| `recall` | 맞힌 수 ÷ 고유 골드 수 |
| `f1` | `2 × TP / (2 × TP + FP + FN)` |

세 문자열이 모두 같아야 골드와 일치합니다. 이름 표기만 다른 경우도 FP와 FN에 나타날 수 있으므로 원문으로 원인을 확인하세요.  

각 공식의 분모가 0이면 `None`을 넣고 미산출로 출력하세요.  
계산값은 반올림하지 않고 저장하고, 출력할 때만 소수 자릿수를 정하세요.  

In [ ]:
# (1) task_rows와 task_gold를 triple_key로 바꿔 각각 task_predicted, task_expected 집합을 만드세요.
# (2) 교집합과 차집합으로 TP, FP, FN 집합을 구해 task_metrics의 tp, fp, fn에 담으세요.
# (3) 같은 집합의 개수로 precision, recall, f1을 계산해 task_metrics에 추가하세요.
# 분모가 0인 지표만 None으로 기록합니다.
# (4) TP, FP, FN 개수와 세 지표를 출력하고, FP와 FN 관계를 정렬해 출력하세요.
# 여기에 코드를 작성하세요.

#### 확인하기

전체 범위, 세 관계 집합과 지표의 분모가 맞는지 확인합니다.  

In [ ]:
# [제공코드]
predicted_keys = {triple_key(row) for row in task_rows}
gold_keys = {triple_key(row) for row in task_gold}
assert len(gold_keys) == 5
assert task_rows == original_rows, "골드 평가에도 전체 저장 행을 사용하세요"
assert task_gold == read_json(data_dir / "assignment_gold.json"), "골드를 줄이지 마세요"
assert task_predicted == predicted_keys
assert task_expected == gold_keys
assert task_metrics["tp"] == predicted_keys & gold_keys
assert task_metrics["fp"] == predicted_keys - gold_keys
assert task_metrics["fn"] == gold_keys - predicted_keys

# 정밀도는 고유 추출 수, 재현율은 고유 골드 수를 분모로 썼는지 확인합니다.
hits = len(predicted_keys & gold_keys)
if predicted_keys:
    assert isclose(task_metrics["precision"], hits / len(predicted_keys))
else:
    assert task_metrics["precision"] is None
assert isclose(task_metrics["recall"], hits / len(gold_keys))
assert isclose(task_metrics["f1"], 2 * hits / (len(predicted_keys) + len(gold_keys)))
print("전체 골드와 추출 범위를 유지했습니다.")

## 4. 청크 크기만 바꾸고 같은 기준으로 비교합니다

`task_before`는 변경 전 저장본입니다. 새 결과 `task_snapshot`과 구분해 유지하세요.  
원문은 493자이므로 1,000자로 바꾸면 전체가 한 청크에 들어갑니다. 실제로는 겹치는 다음 청크가 없습니다.  
큰 청크가 항상 더 좋은 것은 아닙니다. 제품명과 이상반응 문장이 함께 전달되는지 읽고 결과를 비교합니다.  

#### 기준 실행의 설정 유지

기존 저장본을 task_before에 남기고 같은 스키마와 프롬프트를 사용합니다.  

In [ ]:
# [제공코드]

task_before = task_snapshot
schema = task_before["schema"]
prompt_template = task_before["prompt_template"]

#### 재실행 연결과 모델 준비

이 셀부터 실제 모델과 DB를 사용합니다. LV1과 같은 모델을 준비합니다.  

In [ ]:
# [제공코드]

# 현재 작업 폴더부터 상위로 올라가 가장 가까운 .env를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
neo4j_uri = os.environ["NEO4J_URI"]
# driver는 여러 쿼리에서 재사용할 DB 연결 통로입니다. 계정 정보는 출력하지 않습니다.
driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)
# 연결 객체 생성만으로 접속 성공이 보장되지 않으므로 지금 서버 접속을 확인합니다.
driver.verify_connectivity()


def run_cypher(query, **params):
    """값을 매개변수로 전달하고 Cypher 결과를 딕셔너리 목록으로 돌려줍니다."""
    # 쿼리마다 세션을 열고 with 블록이 끝나면 닫습니다. driver는 계속 재사용합니다.
    with driver.session() as session:
        # RETURN에서 붙인 별칭이 딕셔너리 키가 되어 파이썬에서 조회할 수 있습니다.
        return [record.data() for record in session.run(query, **params)]


# 주소에 계정 정보가 포함되어 있어도 호스트와 포트만 확인합니다.
connection_address = urlsplit(neo4j_uri)
print(
    "Neo4j 연결 완료. 호스트:",
    connection_address.hostname,
    "/ 포트:",
    connection_address.port,
)


llm = OpenAILLM(
    model_name="gpt-5.6-luna",  # 관계를 추출할 모델 이름입니다.
)
embedder = OpenAIEmbeddings(
    model="text-embedding-3-large",  # 벡터를 만들 모델 이름입니다.
)
# partial은 이후 embed_query를 호출할 때 dimensions=768을 항상 함께 전달합니다.
embedder.embed_query = partial(embedder.embed_query, dimensions=768)

#### 1000자 청크 확인

설정 사전의 청크 크기를 1000으로 바꾸고 겹침 목표 80은 유지하세요. 원문은 task_doc 그대로입니다.  

In [ ]:
# (1) task_split_settings에 chunk_size=1000, chunk_overlap=80을 담으세요.
# RecursiveCharacterTextSplitter(**task_split_settings)로 task_text_splitter를 만드세요.
# Adapter로 감싸 task_splitter에 담으세요.
# (2) task_doc["text"]를 분할해 task_chunks에 담으세요.
# (3) 각 청크의 순번과 text를 출력해 제품명과 이상반응 내용이 함께 있는지 확인하세요.
# 여기에 코드를 작성하세요.

#### 확인하기

이전 분할기를 재사용하지 않았는지 모델 호출 전에 확인합니다.  

In [ ]:
# [제공코드]
assert task_split_settings == {"chunk_size": 1000, "chunk_overlap": 80}, (
    "청크 크기와 겹침 목표를 확인하세요"
)
# 설정 사전만 맞추고 다른 분할기를 사용한 경우도 찾도록 실제 청크를 대조합니다.
expected_texts = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=80
).split_text(task_doc["text"])
assert [chunk.text for chunk in task_chunks.chunks] == expected_texts, (
    "요구한 설정으로 원문을 다시 분할하세요"
)
print("분할 설정과 실제 청크를 확인했습니다.")

#### 같은 조건으로 다시 추출

새 실행 ID를 사용하고 자동 노드 통합은 계속 끕니다. 이전 실행을 덮어쓰지 않습니다.  

In [ ]:
# (1) 새 task_splitter와 기존 모델, 스키마, 프롬프트로 task_builder를 만드세요.
# schema=deepcopy(schema)로 전달해 이전 실행의 설정이 바뀌지 않게 하세요.
# from_file=False, on_error="RAISE", perform_entity_resolution=False를 유지하세요.
# (2) task_execution_id = str(uuid4())로 새 실행 ID를 만드세요.
# run_async에 text=task_doc["text"], file_path=task_doc["url"]을 전달하세요.
# document_metadata의 source_doc_id는 task_doc["doc_id"], execution_id는 task_execution_id입니다.
# await로 실행한 결과를 task_result에 담으세요.
# (3) task_result.result["writer"]["status"]를 task_writer_status에 담고 출력하세요.
# 여기에 코드를 작성하세요.

#### 실행별 조회 함수

교안의 조회 함수를 그대로 사용합니다. 새 실행의 관계만 읽습니다.  

In [ ]:
# [제공코드]
def read_relations(execution_id):
    """지정한 실행에서 저장한 개체 간 관계를 평가용 사전 목록으로 읽습니다.

    Args:
        execution_id (str): 빌더를 실행할 때 Document에 저장한 실행 ID.
            demo_execution_id처럼, 조회하려는 실행에서 사용한 값을 전달합니다.

    Returns:
        list[dict]: 관계 ID별 트리플과 근거, 출처 정보. 결과가 없으면 [].
            같은 관계의 청크 원문은 chunk_texts 목록에 모읍니다.

    Example:
        반환 형태 예시입니다. 실제 DB 식별자는 다르며 원문은 설명을 위해 줄였습니다.
        [{
            "relationship_id": "관계 식별자 예시",
            "subject": "2.0.3",
            "subject_type": "Release",
            "relation": "FIXES_API",
            "object": "DataFrame.to_string",
            "object_type": "ApiElement",
            "evidence": "Fixed regression when DataFrame.to_string",
            "source_doc_id": "pandas_doc_source_whatsnew_v2_0_3",
            "chunk_texts": ["What's new in 2.0.3 ... Fixed regression when DataFrame.to_string ..."]
        }]
        rows[0]["object"]는 첫 관계의 목적어 이름이며,
        rows[0]["chunk_texts"][0]은 그 관계에 연결된 첫 번째 원문 문자열입니다.
    """
    return run_cypher(
        """
    // (1) 실행 ID로 문서 범위를 고르고, 그 문서의 청크와 주어 개체를 찾습니다.
    MATCH (d:Document {execution_id: $execution_id})
          <-[:FROM_DOCUMENT]-(c:Chunk)<-[:FROM_CHUNK]-(s:__Entity__)
    // (2) 같은 청크에 연결된 목적어를 찾습니다. 관계 타입은 제한하지 않습니다.
    MATCH (s)-[r]->(o:__Entity__)-[:FROM_CHUNK]->(c)
    // (3) AS 오른쪽 이름이 반환 사전의 키가 됩니다.
    RETURN
        elementId(r) AS relationship_id, // DB 안에서 관계를 구분하는 ID입니다.
        s.name AS subject, // 주어 노드의 이름입니다.
        head([x IN labels(s) WHERE NOT x STARTS WITH '__']) AS subject_type, // 관리 레이블을 제외한 첫 타입입니다.
        type(r) AS relation, // 주어에서 목적어로 향하는 관계 타입입니다.
        o.name AS object, // 목적어 노드의 이름입니다.
        head([x IN labels(o) WHERE NOT x STARTS WITH '__']) AS object_type, // 관리 레이블을 제외한 첫 타입입니다.
        coalesce(r.evidence, '') AS evidence, // 근거 인용문이며, 없으면 빈 문자열입니다.
        d.source_doc_id AS source_doc_id, // 원본 문서 ID입니다. 실행 ID와 다릅니다.
        collect(DISTINCT c.text) AS chunk_texts // 연결된 청크 원문을 중복 없이 모읍니다.
    ORDER BY subject, relation, object, relationship_id
    """,
        execution_id=execution_id,
    )


def read_chunks(execution_id):
    """지정한 실행에서 저장한 청크의 순서, 원문, 임베딩 차원 수를 읽습니다.

    Args:
        execution_id (str): 빌더를 실행할 때 Document에 저장한 실행 ID.
            문서 이름이나 source_doc_id가 아니라 demo_execution_id 같은 실행 값을 씁니다.

    Returns:
        list[dict]: 청크별 원문과 임베딩 차원 정보. 순번순으로 정렬하며, 없으면 [].

    Example:
        반환 형태 예시입니다. 원문은 설명을 위해 줄였으며 실제 청크 수와 내용은 다릅니다.
        [
            {"index": 0, "text": "What's new in 2.0.3 ...", "dimensions": 768},
            {"index": 1, "text": "Bug fixes ...", "dimensions": 768}
        ]
    """
    return run_cypher(
        """
    // (1) 지정한 실행의 문서에 연결된 청크만 고릅니다.
    MATCH (c:Chunk)-[:FROM_DOCUMENT]->(d:Document {execution_id: $execution_id})
    // (2) 청크 하나를 사전 하나로 읽습니다. AS 오른쪽이 사전의 키입니다.
    RETURN
        c.index AS index, // 문서 안의 청크 순번입니다. 0부터 시작합니다.
        c.text AS text, // 청크에 저장된 원문입니다.
        size(c.embedding) AS dimensions // 벡터 원소 수, 즉 임베딩 차원입니다.
    // 원문을 읽는 순서대로 확인할 수 있게 청크 순번으로 정렬합니다.
    ORDER BY index
    """,
        execution_id=execution_id,
    )

#### 새 결과를 별도 파일에 저장

large_assignment.json에 분할기에 전달한 설정과 새 결과를 저장합니다. baseline_assignment.json은 유지합니다. 두 파일 모두 통합 전 저장 결과입니다.  

In [ ]:
# [제공코드]
# 실행 ID가 같은 DB 관계와 청크를 각각 읽습니다. 아직 노드를 통합하기 전입니다.
task_rows = read_relations(task_execution_id)
task_stored_chunks = read_chunks(task_execution_id)
print("저장 관계 행 수:", len(task_rows), "/ 청크 수:", len(task_stored_chunks))
for chunk in task_stored_chunks:
    print("청크:", chunk["index"], "/ 임베딩 차원:", chunk["dimensions"])
for row in task_rows:
    print("관계:", row["subject"], "->", row["relation"], "->", row["object"])
    print("근거:", row["evidence"])
    print()

# 문서, 설정과 조회 결과를 함께 저장해 교안 02에서 같은 실행을 평가합니다.
task_snapshot = {
    # 평가 대상이 중복 노드 통합 전 결과임을 기록합니다.
    "stage": "허용하지 않은 노드·관계·속성을 가지치기한 뒤 DB에 저장한 결과(중복 노드 통합 전)",
    # 어느 실행에서 만든 결과인지 구분합니다.
    "execution_id": task_execution_id,
    # 전체 원문: 근거 인용을 검사하고, 같은 문서로 다시 추출할 때 사용합니다.
    "document": task_doc,
    # 실행 설정: 비교 실험에서 바꿀 조건과 유지할 조건을 확인합니다.
    "chunk_size": task_split_settings["chunk_size"],
    "chunk_overlap": task_split_settings["chunk_overlap"],
    "text_splitter": "RecursiveCharacterTextSplitter",
    "model": "gpt-5.6-luna",
    "embedding_model": "text-embedding-3-large",
    "dimensions": 768,
    "schema": schema,
    "prompt_template": prompt_template,
    "perform_entity_resolution": False,
    # 추출 관계: 스키마와 근거를 검사하고 골드와 비교합니다.
    "rows": task_rows,
    # 당시 청크: 원문이 나뉜 위치와 변경 전후의 청크 수와 내용을 확인합니다.
    "chunks": task_stored_chunks,
}
save_json(output_dir / "large_assignment.json", task_snapshot)
print("저장 파일:", output_dir / "large_assignment.json")

#### 반복 평가 함수 준비

3절에서 직접 계산한 집합과 지표를 돌려주는 교안의 함수입니다. 두 실행의 반복 계산에 사용합니다.  

In [ ]:
# [제공코드]
def score_relations(rows, gold):
    """같은 문서 범위의 고유 관계를 골드와 비교합니다.

    Args:
        rows: 평가할 저장 관계 전체. 근거 검사 실패 행도 포함합니다.
        gold: 해당 입력 문서의 고정 정답 목록.
    Returns:
        TP, FP, FN 집합과 정밀도, 재현율, F1을 담은 사전.
    """
    # (1) 같은 관계가 여러 행에 있어도 한 번만 세도록 집합을 만듭니다.
    # 검사 실패 행도 predicted에 포함하고, 골드 목록은 줄이지 않습니다.
    predicted = {triple_key(row) for row in rows}
    expected = {triple_key(row) for row in gold}
    # (2) 맞힘, 잘못 뽑음, 놓침을 집합 비교로 나눕니다.
    tp = predicted & expected  # 양쪽에 있음: 맞힌 관계
    fp = predicted - expected  # 추출에만 있음: 잘못 뽑은 관계
    fn = expected - predicted  # 골드에만 있음: 놓친 관계

    # (3) 정밀도는 추출 수, 재현율은 정답 수로 나눕니다. 분모가 없으면 None입니다.
    precision = len(tp) / len(predicted) if predicted else None
    recall = len(tp) / len(expected) if expected else None
    # 같은 TP, FP, FN으로 F1도 계산합니다.
    denominator = 2 * len(tp) + len(fp) + len(fn)
    f1 = 2 * len(tp) / denominator if denominator else None
    # (4) 점수와 오류 관계를 함께 반환해, 낮은 점수의 원인도 확인할 수 있게 합니다.
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

#### 두 실행의 모든 지표 비교


위에서 준비한 `score_relations`로 같은 계산을 두 실행에 적용하세요.  

- `task_before_metrics`: `task_before["rows"]`와 `task_gold`의 평가 결과.
- `task_after_metrics`: 새 `task_snapshot["rows"]`와 **같은** `task_gold`의 평가 결과.

각 저장본의 전체 행에서 두 검사 비율도 다시 구하고, 300자와 1000자로 구분해 출력하세요.  

In [ ]:
# (1) score_relations로 task_before_metrics와 task_after_metrics를 만드세요.
# 각 저장본의 rows 전체를 사용하고 task_gold는 동일하게 전달하세요.
# (2) 두 실행 각각에서 전체 행의 스키마 준수율, 근거 원문 일치율과 세 지표를 출력하세요.
# 두 검사도 분자와 분모를 함께 출력하고, 행이 없으면 미산출로 표시하세요.
# (3) FP와 FN 목록을 비교해 줄거나 늘어난 오류를 확인하세요.
# 여기에 코드를 작성하세요.

#### 확인하기

청크 크기 외의 조건과 기준 파일을 보존했는지 확인합니다.  

In [ ]:
# [제공코드]
assert task_before["chunk_size"] == 300
assert task_snapshot["chunk_size"] == 1000
assert task_snapshot["chunk_overlap"] == 80
assert read_json(output_dir / "large_assignment.json") == task_snapshot
assert task_before["execution_id"] != task_snapshot["execution_id"]
assert task_writer_status == task_result.result["writer"]["status"] == "SUCCESS"
assert [row["text"] for row in task_snapshot["chunks"]] == [
    chunk.text for chunk in task_chunks.chunks
]
for key in (
    "document",
    "schema",
    "prompt_template",
    "model",
    "embedding_model",
    "dimensions",
    "chunk_overlap",
):
    assert task_before[key] == task_snapshot[key], f"청크 크기 외에 바뀐 설정: {key}"
assert read_json(output_dir / "baseline_assignment.json") == task_before
assert task_gold == read_json(data_dir / "assignment_gold.json")
assert task_before_metrics == score_relations(task_before["rows"], task_gold)
assert task_after_metrics == score_relations(task_snapshot["rows"], task_gold)
print("같은 원문과 골드로 청크 크기만 바꿔 비교했습니다.")
driver.close()

## 결과를 해석합니다

아래 질문에 실행 결과를 근거로 한두 문장씩 답하세요.  

1. 청크 수와 FN 목록이 어떻게 달라졌나요? 점수가 같다면 필요한 문맥이 작은 청크에도 있었는지 확인하세요.  
2. 근거가 원문에 있다는 것만으로 이상반응 관계가 맞다고 볼 수 있나요? 효능의 증상과 구분해 설명하세요.  
3. 점수가 낮아도 골드나 실패 행을 삭제하면 안 되는 이유는 무엇인가요?  

점수가 오르지 않았어도 같은 기준으로 결과를 계산하고 원인을 설명했다면 과제를 올바르게 수행한 것입니다.  